# Glosario de Python para ML
### Qué hace cada método, atributo y función que usamos

---
Este notebook es de **consulta rápida**. Cuando veas algo en el código y no recuerdes qué hace, búscalo aquí.

Organizado por categoría:
- **Sección 1** — Atributos de forma y estructura (shape, dtype, columns...)
- **Sección 2** — Métodos de selección (iloc, loc, drop, filter...)
- **Sección 3** — Métodos de transformación (apply, map, cut, get_dummies...)
- **Sección 4** — Métodos de estadística (describe, corr, mean, sum...)
- **Sección 5** — Funciones de numpy (unique, array, zeros, sqrt...)
- **Sección 6** — Funciones de Python puro (zip, enumerate, range, len...)
- **Sección 7** — Nombres de variables comunes (X, Y, feat, corr...)

In [ ]:
import pandas as pd
import numpy as np

# Dataset de ejemplo para todas las demos de abajo
from sklearn.datasets import load_breast_cancer
dataset = load_breast_cancer()
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
df['diagnostico'] = dataset.target

print("Dataset listo para los ejemplos")
print("Forma:", df.shape)

---
## Sección 1 — Atributos de forma y estructura

Estos NO llevan paréntesis — son propiedades, no métodos.

In [ ]:
# ── .shape ────────────────────────────────────────────────────────────
# Devuelve una tupla (filas, columnas) que describe el tamaño.
# .shape[0] → número de filas (muestras/pacientes)
# .shape[1] → número de columnas (características)
# Funciona en DataFrames y en arrays numpy.
print(".shape del DataFrame completo:", df.shape)
print(".shape[0] = filas:", df.shape[0])
print(".shape[1] = columnas:", df.shape[1])

In [ ]:
# ── .columns ──────────────────────────────────────────────────────────
# Lista con los nombres de todas las columnas del DataFrame.
# Útil cuando no recuerdas cómo se llaman exactamente.
print(".columns:")
print(df.columns.tolist())  # .tolist() lo convierte a lista normal de Python

In [ ]:
# ── .dtypes ───────────────────────────────────────────────────────────
# Muestra el tipo de dato de cada columna.
# float64 = número decimal
# int64   = número entero
# object  = texto (string)
# bool    = verdadero/falso
# Si una columna debería ser número pero es 'object', algo salió mal al cargar.
print(".dtypes (primeras 5 columnas):")
print(df.dtypes.head())

In [ ]:
# ── .index ────────────────────────────────────────────────────────────
# El índice (número de fila) de cada registro.
# Por default es 0, 1, 2, 3, ... N-1
# Después de filtrar o dividir, los índices no se reinician solos.
print(".index:", df.index)
print("Primer índice:", df.index[0])
print("Último índice:", df.index[-1])

In [ ]:
# ── .values ───────────────────────────────────────────────────────────
# Convierte el DataFrame a un array numpy puro (sin nombres de columnas).
# Muchos modelos de ML lo necesitan así.
# Alternativa más moderna: .to_numpy()
print(".values (primeras 2 filas):")
print(df.head(2).values)

---
## Sección 2 — Métodos de selección

Para acceder a partes específicas del DataFrame.

In [ ]:
# ── .head() y .tail() ─────────────────────────────────────────────────
# .head(n) → primeras n filas (default 5)
# .tail(n) → últimas n filas (default 5)
# Casi siempre lo primero que haces después de cargar datos.
print("Primeras 3 filas:")
display(df.head(3))
print("Últimas 2 filas:")
display(df.tail(2))

In [ ]:
# ── .drop() ───────────────────────────────────────────────────────────
# Elimina filas o columnas por su nombre o índice.
# NO es para eliminar el último registro — es para eliminar por NOMBRE.
#
# Parámetros importantes:
#   columns=['nombre'] → elimina esa columna
#   index=[0, 1]       → elimina esas filas por índice
#   inplace=True       → modifica el DataFrame original (sin inplace, crea uno nuevo)
#
# USOS MÁS COMUNES:
#   X = df.drop(columns=['diagnostico'])  → quitar la columna Y para quedarnos con X
#   corr.drop('__Y__')                    → quitar una columna del resultado de corr()

# Ejemplo: quitar la columna 'diagnostico' para obtener solo X
X_solo = df.drop(columns=['diagnostico'])
print("Con drop(columns=['diagnostico']):")
print("  Antes:", df.shape, "→ Después:", X_solo.shape)

In [ ]:
# ── .iloc[] ───────────────────────────────────────────────────────────
# Selección por POSICIÓN (número de fila/columna, como índice de lista)
# iloc = Integer Location
# Sintaxis: df.iloc[fila, columna]
#   Puede ser un número, un rango [inicio:fin], o lista [0, 2, 5]
#
# Ejemplo: df.iloc[0, 2]  → fila 0, columna 2
#          df.iloc[:3, 0] → primeras 3 filas, columna 0
#          df.iloc[:, 0]  → todas las filas, columna 0
print("Fila 0, columna 2 (por posición):")
print(df.iloc[0, 2])
print()
print("Primeras 3 filas, primeras 2 columnas:")
print(df.iloc[:3, :2])

In [ ]:
# ── .loc[] ────────────────────────────────────────────────────────────
# Selección por NOMBRE (nombre de fila/columna)
# loc = Label Location
# Más legible que iloc cuando las columnas tienen nombres.
#
# Ejemplo: df.loc[0, 'mean radius']       → fila 0, columna 'mean radius'
#          df.loc[:, 'mean radius']        → todas las filas de esa columna
#          df.loc[:, ['col1', 'col2']]     → varias columnas a la vez
print("Columna 'mean radius' con loc:")
print(df.loc[:, 'mean radius'].head(3))

In [ ]:
# ── Selección por nombre de columna ───────────────────────────────────
# La forma más común de acceder a una columna.
# df['columna']      → devuelve una Serie (una columna)
# df[['col1','col2']]  → devuelve un DataFrame (varias columnas)
# df[lista_de_cols]    → igual que arriba cuando tienes la lista en variable

# Una columna:
print("Una columna:")
print(df['mean radius'].head(3))
print()

# Varias columnas usando una lista:
cols_elegidas = ['mean radius', 'mean area', 'diagnostico']
print("Varias columnas:")
print(df[cols_elegidas].head(3))

In [ ]:
# ── Filtrado con condiciones ──────────────────────────────────────────
# Para seleccionar filas que cumplan una condición.
# df[condicion] → devuelve solo las filas donde la condición es True

# Ejemplo: solo los pacientes malignos
malignos = df[df['diagnostico'] == 0]
benignos = df[df['diagnostico'] == 1]
print(f"Pacientes malignos (diagnostico=0): {len(malignos)}")
print(f"Pacientes benignos (diagnostico=1): {len(benignos)}")

---
## Sección 3 — Métodos de transformación

In [ ]:
# ── .map() ────────────────────────────────────────────────────────────
# Reemplaza valores usando un diccionario de correspondencias.
# Funciona en Series (columnas individuales), no en DataFrames completos.
#
# USOS COMUNES:
#   Convertir texto a números:  {'M': 0, 'B': 1}
#   Renombrar categorías:       {'Yes': 1, 'No': 0, 'Maybe': 0}
#   Cualquier sustitución 1-a-1
#
# ⚠ Si algún valor del original NO está en el diccionario, se vuelve NaN.
#   Por eso siempre verifica que el mapeo cubra todos los valores únicos.

serie_texto = pd.Series(['M', 'B', 'M', 'B', 'M'])
print("Antes:", serie_texto.values)
print("Después de .map({'M':0, 'B':1}):", serie_texto.map({'M': 0, 'B': 1}).values)

In [ ]:
# ── .apply() ──────────────────────────────────────────────────────────
# Aplica una función a cada columna (o fila) del DataFrame.
# Es como un for loop sobre las columnas, pero más eficiente.
#
# Sintaxis: df.apply(funcion, eje)
#   axis=0 → aplica la función a cada COLUMNA (default)
#   axis=1 → aplica la función a cada FILA
#
# EN EL MACHOTE LO USAMOS PARA BINARIZAR:
#   X_train.apply(pd.cut, bins=2, labels=[0,1])
#   → aplica pd.cut() a cada columna por separado
#   → cada columna se binariza según su propio rango (no mezclado con otras)

X_ejemplo = df.drop(columns=['diagnostico']).iloc[:, :3]  # solo 3 columnas para ver bien
print("Antes (primeras 3 cols, 3 filas):")
print(X_ejemplo.head(3))
print()
binarizado = X_ejemplo.apply(pd.cut, bins=2, labels=[0, 1])
print("Después de apply(pd.cut):")
print(binarizado.head(3))

In [ ]:
# ── pd.cut() ──────────────────────────────────────────────────────────
# Divide una serie numérica en N rangos (bins) y asigna etiquetas.
# 'cut' en inglés = cortar, dividir en rebanadas.
#
# Parámetros importantes:
#   bins=2      → cuántos rangos/grupos crear
#   labels=[0,1] → qué etiqueta asignar a cada rango (de menor a mayor)
#
# CÓMO FUNCIONA:
#   Si los valores van de 6 a 28 y bins=2:
#     Rango bajo: 6–17  → primer label (labels[0])
#     Rango alto: 17–28 → segundo label (labels[1])
#
# ⚠ EL ORDEN DE LOS LABELS IMPORTA:
#   labels=[0,1] → valor bajo=0, valor alto=1 (normal)
#   labels=[1,0] → valor bajo=1, valor alto=0 (invertido, para correlaciones negativas)

valores = pd.Series([6.5, 10.2, 17.0, 22.5, 28.1])
print("Valores originales:", valores.values)
print("Con labels=[0,1]:",   pd.cut(valores, bins=2, labels=[0,1]).values)
print("Con labels=[1,0]:",   pd.cut(valores, bins=2, labels=[1,0]).values)

In [ ]:
# ── .to_numpy() ───────────────────────────────────────────────────────
# Convierte un DataFrame o Serie a un array numpy puro.
# Se usa cuando el modelo exige numpy y no acepta DataFrame directamente.
# Es más moderno que .values (aunque ambos hacen lo mismo).

serie = df['mean radius']
print("Tipo antes:",  type(serie))
print("Tipo después:", type(serie.to_numpy()))
print("Valores:", serie.to_numpy()[:5])

In [ ]:
# ── .copy() ───────────────────────────────────────────────────────────
# Crea una copia independiente del DataFrame.
# IMPORTANTÍSIMO: si no usas .copy(), el nuevo DataFrame es una referencia
# al original, y modificar uno modifica el otro.
#
# REGLA: si vas a modificar un DataFrame, siempre empieza con .copy()

# Sin copy (peligroso):
df_sin_copy = df              # apunta al mismo lugar
df_sin_copy['nueva'] = 99    # ← ESTO TAMBIÉN MODIFICA df ORIGINAL

# Con copy (correcto):
df_con_copy = df.copy()       # copia independiente
df_con_copy['nueva'] = 99    # ← solo modifica df_con_copy, no df

# Limpiar el ejemplo
df = df.drop(columns=['nueva'], errors='ignore')
print("Con .copy() el original queda intacto")

In [ ]:
# ── .isnull() / .isna() ───────────────────────────────────────────────
# Detecta valores faltantes (NaN = Not a Number).
# .isnull() devuelve True/False por cada celda.
# .isnull().sum() cuenta cuántos NaN hay por columna.
# .isnull().any() dice si hay AL MENOS UN NaN por columna.
#
# ⚠ Los NaN rompen casi todos los modelos de ML.
#   Siempre verifica antes de entrenar.

print("NaN por columna (primeras 5):")
print(df.isnull().sum().head())
print()
print("¿Hay algún NaN en todo el dataset?", df.isnull().any().any())

---
## Sección 4 — Métodos de estadística

In [ ]:
# ── .describe() ───────────────────────────────────────────────────────
# Estadísticas básicas de cada columna numérica:
#   count → cuántos valores (si es menor al total, hay NaN)
#   mean  → promedio
#   std   → desviación estándar (qué tan dispersos están los valores)
#   min   → valor mínimo
#   25%   → percentil 25 (el 25% de los datos están por debajo de este valor)
#   50%   → mediana (el valor del medio)
#   75%   → percentil 75
#   max   → valor máximo
#
# Sirve para entender los rangos antes de binarizar o escalar.

print("describe() primeras 3 columnas:")
df.iloc[:, :3].describe()

In [ ]:
# ── .corr() ───────────────────────────────────────────────────────────
# Calcula la correlación entre TODAS las columnas del DataFrame.
# Devuelve una matriz (cuadrada) de correlaciones.
#
# La diagonal siempre es 1.0 (una columna correlaciona perfectamente consigo misma).
#
# USO PARA ENCONTRAR CORRELACIÓN CON Y:
#   df_temp['__Y__'] = Y         → agrega Y como columna
#   df_temp.corr()['__Y__']      → toma solo la fila/columna de Y
#   .drop('__Y__')               → quita la correlación de Y consigo misma (siempre 1.0)

print("Correlación entre las primeras 3 columnas + diagnostico:")
df[['mean radius', 'mean area', 'diagnostico']].corr()

In [ ]:
# ── .sum() ────────────────────────────────────────────────────────────
# Suma los valores. Funciona en arrays, Series y DataFrames.
# En DataFrames, suma cada columna por separado (axis=0, default).
# En la MPNeuron se usa: sum(x) donde x es una fila [0,1,1,0,1,...]
# sum(x) suma todos los valores = cuántos 1s hay.

ejemplo_binario = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1])  # fila de un paciente
print("Fila binaria:", ejemplo_binario)
print("sum(fila):", sum(ejemplo_binario), "← cuántas características son '1'")

In [ ]:
# ── .mean() .min() .max() .std() ──────────────────────────────────────
# Estadísticas básicas de una Serie o columna.
col = df['mean radius']
print(f"mean radius:")
print(f"  Promedio (mean): {col.mean():.2f}")
print(f"  Mínimo   (min):  {col.min():.2f}")
print(f"  Máximo   (max):  {col.max():.2f}")
print(f"  Desv. std (std): {col.std():.2f}")

In [ ]:
# ── .value_counts() ───────────────────────────────────────────────────
# Cuenta cuántas veces aparece cada valor único en una Serie.
# Muy útil para ver la distribución de Y o de columnas categóricas.

print(".value_counts() de diagnostico:")
print(df['diagnostico'].value_counts())
print()
print("Con normalize=True (porcentajes):")
print(df['diagnostico'].value_counts(normalize=True).round(3))

In [ ]:
# ── .sort_values() ────────────────────────────────────────────────────
# Ordena una Serie o DataFrame por sus valores.
# ascending=True → de menor a mayor (default)
# ascending=False → de mayor a menor
# .abs() → valor absoluto (para ordenar por magnitud ignorando el signo)

correlaciones = df.corr()['diagnostico'].drop('diagnostico')
print("5 características MÁS correlacionadas (por magnitud):")
print(correlaciones.abs().sort_values(ascending=False).head())

---
## Sección 5 — Funciones de numpy

In [ ]:
# ── np.unique() ───────────────────────────────────────────────────────
# Devuelve los valores únicos de un array, ordenados.
# Con return_counts=True también devuelve cuántas veces aparece cada uno.
# Se usa mucho para ver las clases en Y.

Y = dataset.target
print("np.unique(Y):", np.unique(Y))
print()

valores, conteos = np.unique(Y, return_counts=True)
print("Con return_counts=True:")
print("  valores:", valores)
print("  conteos:", conteos)

In [ ]:
# ── np.array() ────────────────────────────────────────────────────────
# Crea un array numpy desde una lista de Python.
# Los arrays numpy son más eficientes que las listas para operaciones matemáticas.

lista_python = [1, 0, 1, 1, 0]
array_numpy  = np.array(lista_python)
print("Lista Python:", lista_python, type(lista_python))
print("Array numpy: ", array_numpy,  type(array_numpy))

In [ ]:
# ── np.sqrt() ─────────────────────────────────────────────────────────
# Raíz cuadrada. Se usa para calcular RMSE (raíz del MSE).
# RMSE = np.sqrt(MSE)

print("np.sqrt(9):", np.sqrt(9))
print("np.sqrt(2):", np.sqrt(2))

In [ ]:
# ── np.isnan() ────────────────────────────────────────────────────────
# Detecta valores NaN en un array numpy.
# .any() dice si hay al menos uno.

array_con_nan = np.array([1.0, 2.0, np.nan, 4.0])
print("Array:", array_con_nan)
print("np.isnan():", np.isnan(array_con_nan))
print("¿Hay algún NaN?", np.isnan(array_con_nan).any())

---
## Sección 6 — Funciones de Python puro

In [ ]:
# ── zip() ─────────────────────────────────────────────────────────────
# Combina dos (o más) listas en pares, elemento por elemento.
# No es exactamente como un diccionario — es más como un cierre relámpago (zipper):
# une dos listas en paralelo.
#
# Se usa mucho para iterar sobre valores y conteos al mismo tiempo.
#
# ⚠ Si las listas tienen longitud diferente, zip se detiene en la más corta.

valores = [0, 1]
conteos = [212, 357]

print("Sin zip (dos loops separados):")
for i in range(len(valores)):
    print(f"  Clase {valores[i]}: {conteos[i]}")

print()
print("Con zip (más elegante):")
for v, c in zip(valores, conteos):
    print(f"  Clase {v}: {c}")

In [ ]:
# ── enumerate() ───────────────────────────────────────────────────────
# Itera sobre una lista y te da también el índice (número de posición).
# Útil cuando necesitas el índice Y el valor al mismo tiempo.

nombres = ['radio', 'textura', 'perímetro']

print("Sin enumerate:")
for i in range(len(nombres)):
    print(f"  {i}: {nombres[i]}")

print()
print("Con enumerate (más limpio):")
for i, nombre in enumerate(nombres):
    print(f"  {i}: {nombre}")

In [ ]:
# ── range() ───────────────────────────────────────────────────────────
# Genera una secuencia de números enteros.
# range(n)        → de 0 a n-1
# range(a, b)     → de a a b-1
# range(a, b, paso) → de a a b-1 saltando de 'paso' en 'paso'
#
# En la MPNeuron se usa:
#   for th in range(X.shape[1] + 1)
#   → genera 0, 1, 2, ..., 30  (para probar todos los thresholds posibles)

print("range(5):",       list(range(5)))
print("range(2, 8):",    list(range(2, 8)))
print("range(0, 10, 2):",list(range(0, 10, 2)))

In [ ]:
# ── len() ─────────────────────────────────────────────────────────────
# Devuelve el número de elementos de una lista, array, Serie o DataFrame.
# En DataFrames, len() cuenta las FILAS (no las columnas).
# Para columnas usa .shape[1]

lista = [1, 2, 3, 4, 5]
print("len(lista):", len(lista))
print("len(df) = filas:", len(df))
print("df.shape[1] = columnas:", df.shape[1])

In [ ]:
# ── max() y min() ─────────────────────────────────────────────────────
# max(lista)           → valor máximo
# max(dict, key=func)  → llave del dict donde func es máxima
#
# En la MPNeuron se usa:
#   self.threshold = max(accuracy, key=accuracy.get)
#   → de todas las llaves del dict 'accuracy', devuelve la que tiene el valor más alto
#   → es decir, el threshold con el mejor accuracy

accuracy = {0: 0.62, 1: 0.71, 2: 0.88, 3: 0.85, 4: 0.79}
print("Diccionario de accuracy por threshold:", accuracy)
print("max(accuracy.values()) = mejor accuracy:", max(accuracy.values()))
print("max(accuracy, key=accuracy.get) = threshold óptimo:", max(accuracy, key=accuracy.get))

---
## Sección 7 — Nombres de variables comunes

En ML hay convenciones de nombres que se repiten en casi todo el código.
No son palabras reservadas de Python — son solo costumbres del campo.

In [ ]:
# ── Convenciones de nombres ───────────────────────────────────────────
convenciones = {
    # Variables principales
    'X':           'DataFrame con las características (features). Mayúscula por convención matemática.',
    'Y':           'Array con las etiquetas (lo que queremos predecir). Mayúscula también.',
    'y_pred':      'Predicciones del modelo. Minúscula para diferenciar de Y real.',
    'y_train':     'Etiquetas del conjunto de entrenamiento.',
    'y_test':      'Etiquetas del conjunto de prueba.',
    'X_train':     'Características del conjunto de entrenamiento.',
    'X_test':      'Características del conjunto de prueba.',

    # Nombres de columnas y grupos
    'feat':        'Abreviación de feature (característica). Una sola columna de X.',
    'features':    'Lista de nombres de columnas seleccionadas.',
    'col':         'Abreviación de column (columna).',
    'cols':        'Lista de nombres de columnas.',

    # Estadísticas
    'corr':        'Abreviación de correlation (correlación). Serie o matriz de correlaciones.',
    'acc':         'Abreviación de accuracy (exactitud del modelo).',
    'cm':          'Abreviación de confusion matrix (matriz de confusión).',

    # Modelos y parámetros
    'modelo':      'El objeto del modelo de ML (ej: LogisticRegression())',
    'scaler':      'El objeto que escala los datos (StandardScaler, MinMaxScaler)',
    'th':          'Abreviación de threshold (umbral). En la MPNeuron es el punto de corte.',
    'umbral':      'En español, el valor de corte para filtrar correlaciones.',

    # Iteración
    'i':           'Índice de un loop. Solo vive dentro del for.',
    'v':           'Valor en un loop (de values).',
    'c':           'Conteo en un loop (de count).',
    'k':           'Llave en un loop sobre diccionario (de key).',

    # Sufijos comunes
    '_bin':        'Sufijo que indica datos binarizados. X_train_bin = X_train binarizado.',
    '_sc':         'Sufijo que indica datos escalados. X_train_sc = X_train escalado.',
    '_red':        'Sufijo que indica versión reducida. X_red = X con menos columnas.',
    '_raw':        'Sufijo que indica datos sin procesar. Y_raw = Y antes de convertir.',
}

print("Convenciones de nombres en ML:")
print("=" * 60)
for nombre, descripcion in convenciones.items():
    print(f"\n  {nombre}")
    print(f"    {descripcion}")

---
## Sección 8 — Errores comunes y cómo resolverlos

In [ ]:
errores = {
    "KeyError: 'diagnosis'": (
        "La columna no se llama 'diagnosis' exactamente. "
        "UCI usa 'Diagnosis' con D mayúscula. "
        "Solución: print(Y_raw.columns) para ver el nombre exacto."
    ),
    "ModuleNotFoundError: No module named 'machote_ML'": (
        "Python no encuentra el archivo .py. "
        "Verifica que RUTA_MACHOTE apunte a la CARPETA (no al archivo). "
        "Prueba: import os; print(os.listdir(RUTA_MACHOTE))"
    ),
    "ValueError: Input contains NaN": (
        "Hay valores faltantes en los datos. "
        "Verifica con df.isnull().sum() cuáles columnas tienen NaN. "
        "Soluciones: eliminar esas filas (.dropna()) o rellenarlas (.fillna())"
    ),
    "accuracy muy bajo (cerca del % de la clase mayoritaria)": (
        "El modelo no está aprendiendo nada — solo predice siempre la clase más común. "
        "En Breast Cancer era 62.7%% benigno. Si accuracy ≈ 62%%, algo está mal. "
        "Posibles causas: labels invertidos en binarización, train/test mezclados, "
        "threshold=0 elegido como óptimo."
    ),
    "El path funciona en casa pero no en la escuela": (
        "Porque la ruta absoluta (C:/Users/Diana/...) es diferente en cada computadora. "
        "Solución: usar ruta relativa con os.path. "
        "Ver la celda de 'Importar con ruta relativa' en la plantilla de ejercicio."
    ),
}

print("Errores comunes y soluciones:")
print("=" * 60)
for error, solucion in errores.items():
    print(f"\n❌ {error}")
    print(f"   → {solucion}")